# 1. Import Libraries

In [1]:
import sys
from pathlib import Path

# Thêm thư mục gốc (project root) vào Python path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
import xgboost as xgb

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error, root_mean_squared_error

import numpy as np

from utils.features import build_features

import optuna

# 2. Load Dataset

In [2]:
df_train, df_val = build_features(
                                df_train = pd.read_excel("../Dataset/training.xlsx"), 
                                df_test = None,
                                pls_fit_years = [1]
                                   )

In [3]:
features = [
    'Year', 'Month', 'Day', 'Hour',
    'Combined_Temp',
    'Combined_GHI_1', 'Combined_GHI_2',
    'sin_day_k1', 'cos_day_k1',
    'sin_week_k1', 'cos_week_k1',
]
target = 'Load'

# 3. Split Data

In [4]:
# Year 1 → TRAIN
train_mask = df_train['Year'] == 1

# Year 2 → VALIDATION
val_mask = df_val['Year'] == 2

# Year 3 → TEST
# Testing.xlsx is assumed to contain Year 3

X_train = df_train.loc[train_mask, features]
y_train = df_train.loc[train_mask, target]

X_val = df_val.loc[val_mask, features]
y_val = df_val.loc[val_mask, target]

print("Dataset sizes:")
print(f"Train (Year 1): {len(X_train):,}")
print(f"Validation (Year 2): {len(X_val):,}")

Dataset sizes:
Train (Year 1): 8,784
Validation (Year 2): 8,760


# 4. Train on year 1, evaluate on year 2

In [5]:
tscv = TimeSeriesSplit(n_splits=4)

In [6]:
def objective(trial):

    params = {
        "objective": "reg:squarederror",

        "n_estimators": trial.suggest_int(
            "n_estimators",
            100,
            1500
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.2,
            log=True
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            3,
            10
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight",
            1,
            10
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.6,
            1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.6,
            1.0
        ),

        "gamma": trial.suggest_float(
            "gamma",
            0,
            5
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            1e-8,
            10,
            log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            1e-3,
            10,
            log=True
        ),

        "random_state": 42,
        "n_jobs": -1
    }

    fold_rmse = []

    # ============================================
    # Time-series CV trên YEAR 1
    # ============================================

    for train_idx, val_idx in tscv.split(X_train):

        X_fold_train = X_train.iloc[train_idx]
        X_fold_val = X_train.iloc[val_idx]

        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]

        model = xgb.XGBRegressor(**params)

        model.fit(
            X_fold_train,
            y_fold_train
        )

        y_pred = model.predict(
            X_fold_val
        )

        rmse = root_mean_squared_error(
            y_fold_val,
            y_pred
        )

        fold_rmse.append(rmse)

    # Optuna minimize mean CV RMSE
    return np.mean(fold_rmse)

In [7]:
study = optuna.create_study(
    direction="minimize",
    study_name="xgb_year1_timeseries_cv"
)

study.optimize(
    objective,
    n_trials=100,
    show_progress_bar=True
)

[I 2026-10-02 18:36:52,711] A new study created in memory with name: xgb_year1_timeseries_cv


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-10-02 18:36:53,866] Trial 0 finished with value: 272.34679412841797 and parameters: {'n_estimators': 629, 'learning_rate': 0.029815109648957262, 'max_depth': 3, 'min_child_weight': 4, 'subsample': 0.917874635934167, 'colsample_bytree': 0.7248094579283273, 'gamma': 2.0442054093473256, 'reg_alpha': 5.7211523062052425, 'reg_lambda': 0.11860723765336996}. Best is trial 0 with value: 272.34679412841797.
[I 2026-10-02 18:36:57,115] Trial 1 finished with value: 294.3945655822754 and parameters: {'n_estimators': 513, 'learning_rate': 0.01926439609105531, 'max_depth': 9, 'min_child_weight': 1, 'subsample': 0.8823406933870284, 'colsample_bytree': 0.7451077678539537, 'gamma': 3.8181121182157964, 'reg_alpha': 7.653299667890859, 'reg_lambda': 0.012200033686115976}. Best is trial 0 with value: 272.34679412841797.
[I 2026-10-02 18:37:02,268] Trial 2 finished with value: 295.27698516845703 and parameters: {'n_estimators': 1355, 'learning_rate': 0.04018058471825153, 'max_depth': 9, 'min_child_w

In [8]:
print("Best CV RMSE:")
print(study.best_value)

print("\nBest parameters:")

for param, value in study.best_params.items():
    print(f"{param}: {value}")

Best CV RMSE:
260.5398941040039

Best parameters:
n_estimators: 1319
learning_rate: 0.031196340478036013
max_depth: 3
min_child_weight: 6
subsample: 0.7657530194390537
colsample_bytree: 0.6349189400332539
gamma: 1.600631021482472
reg_alpha: 0.8179299007457254
reg_lambda: 0.0875513743610601


In [9]:
best_params = study.best_params

seeds = [0, 1, 2, 3, 42]

results = []

for seed in seeds:

    model = xgb.XGBRegressor(
        **best_params,
        objective="reg:squarederror",
        random_state=seed,
        n_jobs=-1
    )

    model.fit(
        X_train,
        y_train
    )

    y_pred = model.predict(X_val)

    results.append({
        "Seed": seed,
        "RMSE": root_mean_squared_error(y_val, y_pred),
        "MSE": mean_squared_error(y_val, y_pred),
        "MAE": mean_absolute_error(y_val, y_pred),
        "MAPE": mean_absolute_percentage_error(y_val, y_pred),
        "R2": r2_score(y_val, y_pred)
    })

stability_df = pd.DataFrame(results)

stability_summary = stability_df.drop(
    columns="Seed"
).agg(["mean", "std", "min", "max"]).T

print("=== Results by seed ===")
display(stability_df)

print("\n=== Stability summary ===")
display(stability_summary)

=== Results by seed ===


,Seed,RMSE,MSE,MAE,MAPE,R2
0,0,161.711685,26150.669922,117.094360,0.055096,0.841711
1,1,161.825943,26187.636719,117.093147,0.055045,0.841487
2,2,161.207214,25987.763672,116.832954,0.054965,0.842697
3,3,160.559326,25779.296875,115.837906,0.054477,0.843959
4,42,161.324585,26025.621094,116.609688,0.054881,0.842468



=== Stability summary ===


,mean,std,min,max
RMSE,161.325751,0.500149,160.559326,161.825943
MSE,26026.197656,161.240050,25779.296875,26187.636719
MAE,116.693611,0.519355,115.837906,117.094360
MAPE,0.054893,0.000246,0.054477,0.055096
R2,0.842465,0.000976,0.841487,0.843959


In [10]:
stability_df.to_csv("../report/seed_weekly_first_harmonic_val_report.csv", index = False)
stability_summary.to_csv("../report/seed_summary_weekly_first_harmonic_val_report.csv", index = False)

# 4. Final Training on year 1 and year 2

In [11]:
df_train, df_test = build_features(
                                df_train = pd.read_excel("../Dataset/training.xlsx"), 
                                df_test = pd.read_excel("../Dataset/testing.xlsx")
                                   )

In [12]:
X_train = df_train[features]
y_train = df_train[target]

X_test = df_test[features]

In [13]:
print("Dataset sizes:")
print(f"Train (Year 1): {len(X_train):,}")
print(f"Validation (Year 2): {len(X_val):,}")
print(f"Test (Year 3): {len(X_test):,}")

Dataset sizes:
Train (Year 1): 17,544
Validation (Year 2): 8,760
Test (Year 3): 8,760


In [14]:
seeds = [0, 1, 2, 3, 42]

test_predictions = []

for seed in seeds:

    model = xgb.XGBRegressor(
        **best_params,
        objective="reg:squarederror",
        random_state=seed,
        n_jobs=-1
    )

    model.fit(
        X_train,
        y_train
    )

    pred = model.predict(X_test)

    test_predictions.append(pred)

y_test_pred = np.mean(
    test_predictions,
    axis=0
)

In [15]:
pd.DataFrame(y_test_pred).to_csv("../report/weekly_first_harmonic_test_prediction.csv", index = False)